# ClaimLens — Phase 2: LoRA fine-tuning (SFT) on Kaggle

Fine-tunes **Qwen2.5-VL-3B-Instruct** on the 300 synthetic training documents, then re-runs the **exact same**
test and fairness evaluation as the Phase 1 baseline, so the before/after comparison is fair.

**What gets trained:** a LoRA adapter (~30M trainable numbers, about 1% of the model) on the language model only.
The vision encoder and the original 3B weights are frozen.

### Setup (same as Phase 1)
- **Settings:** Accelerator `GPU T4 x2` (training uses one GPU), Internet **On**
- **Input:** your `claimlens-code` dataset, **updated to the latest zip** (New Version)

### Recommended way to run
1. **Dry run first (interactive, about 10 minutes):** keep `QUICK_TEST = True` and run all cells. It trains on
   16 examples and evaluates 3 documents. This proves every cell works before you spend GPU hours.
2. **Real run (background, about 3–3.5 hours):** set `QUICK_TEST = False`, then **Save Version → Save & Run All
   (Commit)**. Training takes about 1 hour; each 100-document evaluation takes about 50 minutes.

In [ ]:
QUICK_TEST = True          # True = 10-minute dry run. False = the real thing.

UPLOADED = "/kaggle/input"
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
ADAPTER_NAME = "sft_v1"

# Training hyper-parameters (see GUIDE.md, Phase 2, for what each one means)
EPOCHS = 2
LEARNING_RATE = 1e-4
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
GRAD_ACCUM = 8                # effective batch size = 8 documents per weight update
POOR_SHARE = 0.35             # oversample poor scans to ~35% of training examples
MAX_TRAIN_MINUTES = 150       # safety stop
RUN_FAIRNESS = True

# Phase 1 test-split hash: confirms we evaluate on exactly the same documents
BASELINE_TEST_HASH_PREFIX = "10005cb99375adfb"

In [ ]:
import os, shutil, subprocess, sys, glob
os.chdir("/kaggle/working")
# The dataset may hold several project copies (old uploads); use the one that has the Phase 2 code.
hits = glob.glob(f"{UPLOADED}/**/claimlens/sft.py", recursive=True)
assert hits, "Phase 2 code not found -- upload the new zip as a New Version of the dataset"
if len(hits) > 1:
    print("Several copies found, using the most recently modified:", *hits, sep="
  ")
src = os.path.dirname(os.path.dirname(max(hits, key=os.path.getmtime)))
print("Using project at:", src)
shutil.rmtree("claimLens", ignore_errors=True)
shutil.copytree(src, "claimLens")
os.chdir("/kaggle/working/claimLens")
sys.path.insert(0, os.getcwd())
assert os.path.exists("claimlens/sft.py"), "Old project version -- upload the new zip as a New Version of the dataset"
print(os.getcwd())

In [ ]:
!pip install -q -U "transformers>=4.49" accelerate "pydantic>=2" peft
# Kaggle ships an old torchao that new peft versions refuse to import alongside. We don't use it.
!pip uninstall -y -q torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Data
Same generator, same seed as Phase 1, so the test documents are identical. The hash check proves it.

In [ ]:
!python -m claimlens generate --out data --train 300 --val 50 --test 100 --fair-sets 25 --fair-variants 4 > /dev/null
import json
info = json.load(open("data/dataset_info.json"))
print("fonts:", info["fonts_used"])
print("test hash:", info["split_hashes"]["test"][:16])
if not info["split_hashes"]["test"].startswith(BASELINE_TEST_HASH_PREFIX):
    print("WARNING: test set differs from the Phase 1 baseline -- comparisons would not be like-for-like!")
else:
    print("OK: identical test set to the Phase 1 baseline")

In [ ]:
from claimlens import sft
train_ex = sft.build_examples("data", "train", poor_share=POOR_SHARE, seed=0)
val_ex = sft.build_examples("data", "val", poor_share=None, seed=0)[:20]
if QUICK_TEST:
    train_ex, val_ex = train_ex[:16], val_ex[:4]
print("train:", sft.describe_examples(train_ex))
print("val:  ", sft.describe_examples(val_ex))
print("\nOne training target (this is exactly what the model learns to write):\n")
print(train_ex[0]["target"])

## 2. Load the base model and attach LoRA

In [ ]:
import torch, time
from transformers import AutoModelForImageTextToText, AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map={"": 0})
BASE_REVISION = getattr(model.config, "_commit_hash", None) or "unknown"
model = sft.apply_lora(model, r=LORA_R, alpha=LORA_ALPHA, dropout=LORA_DROPOUT)
print("base revision:", BASE_REVISION)
print("parameters:", sft.count_parameters(model))
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## 3. Train
Watch `train_loss` fall. `val_loss` (on documents the model never trains on) should fall too. If train loss keeps falling but val loss rises, the model is memorising rather than learning (**overfitting**).

In [ ]:
val_before = sft.evaluate_loss(model, processor, val_ex)
print(f"val loss before training: {val_before:.4f}")
t0 = time.time()
history = sft.train(model, processor, train_ex, val_ex, epochs=1 if QUICK_TEST else EPOCHS, lr=LEARNING_RATE,
                    grad_accum=2 if QUICK_TEST else GRAD_ACCUM, max_minutes=MAX_TRAIN_MINUTES,
                    log_every=1 if QUICK_TEST else 5, out_dir=f"adapters/{ADAPTER_NAME}")
train_minutes = round((time.time() - t0) / 60, 1)
print(f"training took {train_minutes} min; peak GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

In [ ]:
import matplotlib.pyplot as plt
steps = [h for h in history if "train_loss" in h]
epochs_end = [h for h in history if h.get("end_of_epoch")]
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot([h["update"] for h in steps], [h["train_loss"] for h in steps], color="#2a78d6", lw=2, label="train loss")
if epochs_end and "val_loss" in epochs_end[0]:
    per_epoch = max(1, steps[-1]["update"] // len(epochs_end)) if steps else 1
    ax.plot([0] + [per_epoch * (i + 1) for i in range(len(epochs_end))], [val_before] + [h["val_loss"] for h in epochs_end],
            color="#eb6834", lw=2, marker="o", label="val loss")
ax.set_xlabel("weight update"); ax.set_ylabel("loss"); ax.legend(frameon=False)
ax.set_title("SFT training curve", loc="left"); ax.grid(alpha=0.3)
plt.show()

## 4. Save the adapter with a training card
The card records everything needed to reproduce or audit this model: base model and revision, data hashes, hyper-parameters, prompt version and the adapter's own fingerprint.

In [ ]:
from claimlens.prompts import PROMPT_VERSION
from claimlens.evaluation import EVALUATOR_VERSION
card = {
    "adapter_name": ADAPTER_NAME, "base_model": MODEL_ID, "base_revision": BASE_REVISION,
    "method": "LoRA SFT (language model only, vision frozen)",
    "lora": {"r": LORA_R, "alpha": LORA_ALPHA, "dropout": LORA_DROPOUT, "target": sft.LORA_TARGET_REGEX},
    "training": {"epochs": EPOCHS, "lr": LEARNING_RATE, "grad_accum": GRAD_ACCUM, "poor_share": POOR_SHARE,
                 "minutes": train_minutes, "quick_test": QUICK_TEST, "gpu": torch.cuda.get_device_name(0)},
    "data": {"train": sft.describe_examples(train_ex), "val": sft.describe_examples(val_ex),
             "split_hashes": info["split_hashes"], "generator_version": info["generator_version"]},
    "prompt_version": PROMPT_VERSION, "evaluator_version": EVALUATOR_VERSION,
    "val_loss_before": round(val_before, 4), "history_tail": history[-3:],
}
FINGERPRINT = sft.save_adapter(model, f"adapters/{ADAPTER_NAME}", history=history, card=card)
print("adapter fingerprint:", FINGERPRINT)
!ls -lh adapters/{ADAPTER_NAME}

## 5. Switch to inference and sanity-check one test document
Compare with the Phase 1 output for `test-00000`: the boxes should now sit on the values, not on the labels.

In [ ]:
from pathlib import Path
from claimlens.predictors import HFVisionPredictor
from claimlens.runner import run_predictions, evaluate_run

model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True
model = model.merge_and_unload()   # fold the adapter into the weights: faster generation, same answers
pred = HFVisionPredictor.from_model(model, processor, MODEL_ID, f"{BASE_REVISION}+adapter:{ADAPTER_NAME}@{FINGERPRINT}")

label = json.loads(Path("data/test/labels/test-00000.json").read_text())
out = pred.predict(Path("data/test") / label["image"], label)
print(out["latency_ms"], "ms,", out["output_tokens"], "tokens\n")
print(out["raw_text"])

## 6. Evaluate: identical test split, prompt and evaluator as the baseline

In [ ]:
limit = 3 if QUICK_TEST else None
tag = "quick_" if QUICK_TEST else ""
test_dir = run_predictions(pred, "data", "test", runs_dir="runs", run_id=f"{tag}{ADAPTER_NAME}_v2_greedy_test", limit=limit)
evaluate_run(test_dir)
print(open(f"{test_dir}/report.md").read())

In [ ]:
if RUN_FAIRNESS:
    fair_dir = run_predictions(pred, "data", "fairness", runs_dir="runs",
                               run_id=f"{tag}{ADAPTER_NAME}_v2_greedy_fairness", limit=limit)
    evaluate_run(fair_dir)
    print(open(f"{fair_dir}/report.md").read())

## 7. Package results
Download `sft_results.zip` from the **Output** tab and unzip it into `C:\projects\claimLens`. It adds `runs/sft_v1_*` and `adapters/sft_v1` next to your baseline runs.

In [ ]:
!cd /kaggle/working/claimLens && zip -qr /kaggle/working/sft_results.zip runs adapters && ls -lh /kaggle/working/sft_results.zip